# 1.3 — Continuous Diffusion (DDPM) from Scratch

**Mechanism of the day:** destroy data with noise on a schedule, learn to undo **one**
step, then undo a few hundred of them and watch structure appear out of static.

Diffusion has a reputation for being mathematically heavy. It isn't. The entire
method is three equations and a training loop, and by the end of this notebook you
will have written all of them — no `diffusers`, no U-Net, no GPU.

**The forward process** takes real data and drowns it in Gaussian noise over `T`
steps until nothing is left:

```
q(x_t | x_{t-1}) = N( sqrt(1 - beta_t) * x_{t-1},  beta_t * I )
```

The first crucial trick: you never have to *run* that loop, because the composition
of Gaussians is Gaussian. You can jump to any `t` in closed form:

```
q(x_t | x_0) = N( sqrt(abar_t) * x_0,  (1 - abar_t) * I )
    where abar_t = prod_{s<=t} (1 - beta_s)
```

**The reverse process** is what we learn. Reversing noise is hopeless in general —
but `q(x_{t-1} | x_t, x_0)` (reversing *one small step*, knowing where you started)
is a tractable Gaussian. So we train a network to estimate the one thing we're
missing, and the whole thing becomes a sequence of easy problems instead of one
impossible one.

**The objective** is almost insultingly simple. Take clean data, pick a random `t`,
add the corresponding noise, and ask the network which noise you added:

```
loss = || eps  -  eps_theta(x_t, t) ||^2
```

That's it. That's diffusion.

**Our data:** the **Ramachandran plot** — the `(phi, psi)` backbone dihedral angles
of a residue. It is genuinely 2D, so you can plot every sample and see exactly what
the model has learned, but it is not a made-up toy: it is the distribution that
decides whether a residue sits in an alpha-helix or a beta-sheet. Three basins,
one of which holds only ~10% of the mass — so we can also ask the question that
kills GANs: **does the model cover the rare mode, or quietly drop it?**

**How to use this notebook:** read the concept, implement the reps, make the
checkpoints pass. Solutions at the bottom. Trains in ~10s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

# Ramachandran basins: (name, (phi, psi) centre, (sd_phi, sd_psi), weight)
BASINS = [('alpha-helix', (-63.0, -43.0), (12.0, 12.0), 0.45),
          ('beta-sheet',  (-120.0, 130.0), (25.0, 22.0), 0.45),
          ('left-handed', (57.0, 40.0),   (12.0, 12.0), 0.10)]
SCALE = 180.0          # degrees -> roughly [-1, 1], which is what diffusion wants

def make_data(n):
    w = np.array([b[3] for b in BASINS])
    which = rng.choice(len(BASINS), size=n, p=w)
    out = np.zeros((n, 2))
    for i, (_, mu, sd, _) in enumerate(BASINS):
        m = which == i
        k = int(m.sum())
        out[m, 0] = rng.normal(mu[0], sd[0], k)
        out[m, 1] = rng.normal(mu[1], sd[1], k)
    return out.astype(np.float32), which

raw, lab = make_data(8000)
x0_all = torch.from_numpy(raw / SCALE)

def rama(ax, pts_deg, title, color=BLUE, s=4, alpha=.25):
    ax.scatter(pts_deg[:, 0], pts_deg[:, 1], s=s, alpha=alpha, color=color, lw=0)
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
    ax.set_xlabel('phi'); ax.set_ylabel('psi'); ax.set_title(title, fontsize=10)
    ax.grid(alpha=.15)

fig, ax = plt.subplots(figsize=(4.2, 4))
rama(ax, raw, 'the data: Ramachandran (phi, psi)')
for name, mu, _, w in BASINS:
    ax.annotate('%s\n%.0f%%' % (name, 100 * w), mu, fontsize=8, color=INK,
                ha='center', fontweight='bold')
plt.show()
print('data', tuple(x0_all.shape), '| normalized range [%.2f, %.2f]'
      % (x0_all.min(), x0_all.max()))

## Part 1 — the noise schedule

`beta_t` says how much variance to inject at step `t`. Small at first (barely touch
the data), larger later (finish it off). From `betas` we precompute two things we
will use constantly:

- `alphas = 1 - betas` — how much signal *survives* one step
- `abar_t = prod alphas` — how much signal survives from `0` all the way to `t`

The one property the schedule must satisfy: **`abar_T` has to be essentially zero**,
so that `x_T` really is indistinguishable from pure `N(0, I)` noise. If it isn't,
sampling starts from the wrong distribution and everything downstream is subtly
broken. Our checkpoint asserts it.

(We use `T = 200` and a linear schedule. The famous DDPM paper used `T = 1000`;
with fewer steps you need proportionally larger `beta`s to fully destroy the signal.)

### Rep 1 — `make_schedule(T, b0, bT)`
Return `(betas, alphas, abar)`, each a 1-D tensor of length `T`. Linear `betas` from
`b0` to `bT`; `alphas = 1 - betas`; `abar` the **cumulative product** of `alphas`.

In [ ]:
T = 200

def make_schedule(T=T, b0=1e-4, bT=0.06):
    '''Linear beta schedule -> (betas, alphas, abar).'''
    # YOUR CODE HERE
    # hint: torch.linspace and torch.cumprod
    raise NotImplementedError

# --- checkpoint ---
betas, alphas, abar = make_schedule()
assert betas.shape == (T,) and abar.shape == (T,)
assert torch.all(abar[1:] < abar[:-1]), 'abar must strictly decrease'
assert abar[0] > 0.99, 'almost no signal is destroyed on the first step'
assert abar[-1] < 0.01, 'x_T must be essentially pure noise'
print('abar[0] = %.4f  ->  abar[T-1] = %.5f' % (abar[0], abar[-1]))
print('signal surviving to the end: sqrt(abar_T) = %.3f  (i.e. none) ✓' % abar[-1].sqrt())

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(abar.numpy(), color=BLUE, lw=2, label=r'$\bar\alpha_t$ (signal kept)')
ax.plot((1 - abar).numpy(), color=GREEN, lw=2, label=r'$1-\bar\alpha_t$ (noise added)')
ax.set_xlabel('t'); ax.set_ylabel('variance share'); ax.grid(alpha=.15)
ax.set_title('the schedule hands the signal over to noise'); ax.legend(frameon=False, fontsize=9)
plt.show()

## Part 2 — the forward process, in one jump

Here is the trick that makes diffusion trainable. To train on timestep `t` we need
`x_t`, and the naive route is to simulate `t` noising steps. That would make each
training example cost `O(t)` and the whole method impractical.

Instead, the closed form:

```
x_t = sqrt(abar_t) * x_0  +  sqrt(1 - abar_t) * eps        eps ~ N(0, I)
```

One multiply, one add, any `t`, `O(1)`. Note the structure: it is just a weighted
blend of the data and pure noise, with the schedule setting the mix.

The checkpoint is worth reading — it verifies the identity **empirically**, by
comparing the closed-form jump against actually simulating `t` individual steps and
checking the two give the same distribution.

### Rep 2 — `q_sample(x0, t, noise)`
Jump straight to `x_t`. `x0` is `[B, 2]`, `t` a `[B]` tensor of timesteps, `noise`
the same shape as `x0`. Careful with broadcasting: `abar[t]` is `[B]` and needs a
trailing axis to multiply `[B, 2]`.

In [ ]:
def q_sample(x0, t, noise):
    '''Closed-form forward diffusion: x_t given x_0.'''
    # YOUR CODE HERE
    # hint: a = abar[t].unsqueeze(-1)  ->  a.sqrt() * x0 + (1 - a).sqrt() * noise
    raise NotImplementedError

# --- checkpoint ---
x0s = x0_all[:4000]
tt = 60
closed = q_sample(x0s, torch.full((len(x0s),), tt), torch.randn_like(x0s))

# simulate the same number of SINGLE steps and compare the distributions
step = x0s.clone()
for s in range(tt + 1):
    step = alphas[s].sqrt() * step + betas[s].sqrt() * torch.randn_like(step)

assert closed.shape == x0s.shape
assert abs(closed.std() - step.std()) < 0.03, 'closed form must match the simulation'
assert abs(closed.mean() - step.mean()) < 0.03
print('closed form   mean %+.4f  std %.4f' % (closed.mean(), closed.std()))
print('step-by-step  mean %+.4f  std %.4f' % (step.mean(), step.std()))
print('identical distributions, but the closed form is O(1) instead of O(t) ✓')

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, t_show in zip(axes, [0, 25, 60, 150]):
    xt = q_sample(x0_all, torch.full((len(x0_all),), t_show), torch.randn_like(x0_all))
    rama(ax, xt.numpy() * SCALE, 't = %d' % t_show, s=3, alpha=.18)
plt.tight_layout(); plt.show()
print('structure dissolves left to right. Training teaches a net to walk this backwards.')

## Part 3 — the objective: predict the noise

The network takes a noisy point and its timestep, and outputs its guess at **the
noise that was added**: `eps_theta(x_t, t)`. Loss is plain MSE against the true
`eps`. Two questions people always ask:

**Why predict the noise instead of the clean data?** They are algebraically
equivalent — given `x_t` and `eps` you can solve for `x_0`. But `eps` is always
unit-variance regardless of `t`, so the regression target is equally scaled at every
timestep and training is far better conditioned. It also connects diffusion to
**score matching**: the score `grad_x log q(x_t)` equals `-eps / sqrt(1 - abar_t)`,
so a noise predictor *is* a score estimator, and the reverse process is gradient
ascent on log-density. (That equivalence is what rung 2.2's guidance exploits.)

**Why does the network need `t`?** Because the right answer depends on how noisy the
input is. At `t = 5` the point is nearly clean and the noise is a small perturbation;
at `t = 190` the input is almost pure noise. Same input, different correct output —
so `t` must be an input. We inject it with a sinusoidal embedding (the same
positional-encoding idea you used in 1.1, applied to time).

In [ ]:
def timestep_embedding(t, dim=64):
    '''Sinusoidal features for t — same idea as positional encoding in 1.1.'''
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32) / half)
    a = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(a), torch.sin(a)], dim=-1)

class EpsNet(nn.Module):
    '''Tiny MLP: (x_t, t) -> predicted noise. No U-Net needed in 2-D.'''
    def __init__(self, d=128, tdim=64):
        super().__init__()
        self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, d), nn.SiLU(), nn.Linear(d, d))
        self.inp = nn.Linear(2, d)
        self.h1 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.h2 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.out = nn.Sequential(nn.SiLU(), nn.Linear(d, 2))
    def forward(self, x, t):
        h = self.inp(x) + self.tmlp(timestep_embedding(t, self.tdim))   # condition on t
        h = h + self.h1(h)
        h = h + self.h2(h)
        return self.out(h)

model = EpsNet()
print('parameters:', sum(p.numel() for p in model.parameters()))

### Rep 3 — `ddpm_loss(model, x0)`
The whole training objective in four lines: sample `t` uniformly in `[0, T)`, sample
`eps ~ N(0, I)`, build `x_t` with `q_sample`, and return the MSE between the model's
prediction and `eps`.

In [ ]:
def ddpm_loss(model, x0):
    '''Sample a random t, noise x0 to x_t, and score the model's noise prediction.'''
    # YOUR CODE HERE
    # hint: t = torch.randint(0, T, (len(x0),)); noise = torch.randn_like(x0)
    raise NotImplementedError

# --- checkpoint ---
l = ddpm_loss(model, x0_all[:512])
assert l.dim() == 0, 'loss must be a scalar'
assert l.requires_grad, 'loss must be differentiable'
assert 0.5 < l.item() < 2.0, 'an untrained net should score ~1.0 (eps is unit variance)'
print('untrained loss %.3f — about 1.0, because guessing zero gives MSE = Var(eps) = 1 ✓'
      % l.item())

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
hist, t0 = [], time.time()
for step in range(1, 5001):
    ix = torch.randint(0, len(x0_all), (256,))
    loss = ddpm_loss(model, x0_all[ix])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0 or step == 1:
        hist.append((step, loss.item()))
        if step % 1250 == 0 or step == 1:
            print('step %4d  loss %.4f' % (step, loss.item()))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2)
ax.axhline(1.0, color=INK, ls=':', lw=1.5)
ax.text(4900, 1.0, 'predicting zero', ha='right', va='bottom', fontsize=9, color=INK)
ax.set_xlabel('step'); ax.set_ylabel('MSE on the noise'); ax.grid(alpha=.15)
ax.set_title('learning to name the noise')
plt.show()
print('The loss floor is NOT zero: at large t the input is nearly pure noise and the')
print('true eps is genuinely unrecoverable. Diffusion losses always plateau well above 0.')

## Part 4 — the reverse process

Now we walk backwards. Given `x_t` and the predicted noise, the mean of the reverse
step is

```
mu = ( x_t  -  beta_t / sqrt(1 - abar_t) * eps_theta )  /  sqrt(alpha_t)
```

and then we add fresh noise `sigma_t * z` — except at `t = 0`, where we stop and
return the mean (adding noise to your final answer would just blur it).

**Which `sigma_t`?** The DDPM paper offers two, and the choice is visible in your
plots:

- `sigma_t^2 = beta_t` — simple, but slightly **over-disperses** the samples
- `sigma_t^2 = beta_t * (1 - abar_{t-1}) / (1 - abar_t)` — the true posterior
  variance, which is what we use

That second one is smaller (it accounts for how much you already know about
`x_{t-1}`), and it produces basins whose spread matches the real data instead of
being inflated. Small detail, real difference.

### Rep 4 — `p_sample_step(model, xt, t)`
One reverse step, under `torch.no_grad()`. Build the mean with the formula above,
return it unchanged when `t == 0`, otherwise add `post_var[t].sqrt() * z`.

In [ ]:
# posterior variance: beta_tilde_t = beta_t * (1 - abar_{t-1}) / (1 - abar_t)
abar_prev = torch.cat([torch.ones(1), abar[:-1]])
post_var = betas * (1 - abar_prev) / (1 - abar)

@torch.no_grad()
def p_sample_step(model, xt, t):
    '''One denoising step: x_t -> x_{t-1}. t is a python int.'''
    # YOUR CODE HERE
    # hint: tb = torch.full((len(xt),), t, dtype=torch.long) to feed the net
    #       mean = (xt - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
    raise NotImplementedError

# --- checkpoint ---
xt = torch.randn(64, 2)
out = p_sample_step(model, xt, 100)
assert out.shape == xt.shape
a = p_sample_step(model, xt, 0)
b = p_sample_step(model, xt, 0)
assert torch.allclose(a, b), 'the final step (t=0) must be deterministic — no noise added'
c = p_sample_step(model, xt, 100)
assert not torch.allclose(out, c), 'steps with t>0 must be stochastic'
print('one reverse step ✓ — stochastic while t > 0, deterministic at t = 0')

### Rep 5 — `sample_loop(model, n, keep=())`
Start from `x_T ~ N(0, I)` and apply `p_sample_step` for `t = T-1 ... 0`. Return the
final `[n, 2]` sample plus a dict of snapshots for any `t` listed in `keep` (so we
can watch the trajectory).

In [ ]:
@torch.no_grad()
def sample_loop(model, n, keep=()):
    '''Full ancestral sampling from pure noise back to data.'''
    # YOUR CODE HERE
    # hint: x = torch.randn(n, 2); for t in reversed(range(T)): x = p_sample_step(...)
    raise NotImplementedError

# --- checkpoint ---
t0 = time.time()
gen, snaps = sample_loop(model, 4000, keep=(80, 40, 15, 0))
assert gen.shape == (4000, 2)
assert set(snaps) == {80, 40, 15, 0}, 'snapshots missing'
assert gen.abs().max() < 3.0, 'samples should live near the data, not fly off'
print('sampled 4000 points in %.1fs (%d net evaluations)' % (time.time() - t0, T))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
rama(axes[0], torch.randn(4000, 2).numpy() * SCALE, 'x_T (pure noise)', s=3, alpha=.15)
for ax, t_show in zip(axes[1:], [80, 40, 15, 0]):
    rama(ax, snaps[t_show].numpy() * SCALE, 't = %d' % t_show, s=3, alpha=.18)
plt.tight_layout(); plt.show()
print('Noise -> basins. You wrote every line of that. ✓')

## Part 5 — did it cover the modes?

Pictures are persuasive and unreliable. The failure mode we actually care about is
**mode dropping**: a generator that nails the two big basins and quietly ignores the
rare one still looks great in a scatter plot, and would be useless for design — the
rare conformations are often exactly the interesting ones.

So measure it. Assign every sample to its nearest basin centre and compare the
occupancy fractions with the real data. Our left-handed basin holds only 10% of the
mass, which is where a GAN would typically collapse.

### Rep 6 — `basin_fractions(pts_deg)`
Assign each point (in **degrees**) to the nearest of the three basin centres and
return the fraction in each, as a length-3 array summing to 1.

In [ ]:
def basin_fractions(pts_deg):
    '''Fraction of points nearest to each basin centre.'''
    # YOUR CODE HERE
    # hint: cen = np.array([b[1] for b in BASINS]); broadcast to [N, 3, 2],
    #       square-sum over the last axis, argmin, then np.bincount(..., minlength=3)
    raise NotImplementedError

# --- checkpoint ---
gen_deg = gen.numpy() * SCALE
fr_real, fr_gen = basin_fractions(raw), basin_fractions(gen_deg)
assert fr_gen.shape == (3,) and abs(fr_gen.sum() - 1) < 1e-9
assert np.abs(fr_real - fr_gen).max() < 0.10, 'occupancies should roughly match the data'
assert fr_gen[2] > 0.04, 'the rare left-handed mode must NOT be dropped'

print('basin          real   model')
for (name, _, _, _), a, b in zip(BASINS, fr_real, fr_gen):
    print('%-13s %.3f  %.3f' % (name, a, b))
print('\nrare mode kept at %.1f%% of samples — no mode collapse ✓' % (100 * fr_gen[2]))

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.2))
rama(axes[0], raw, 'real data', color=GREEN)
rama(axes[1], gen_deg, 'generated by your DDPM', color=BLUE)
plt.tight_layout(); plt.show()

## Reflection — what just transferred

- **Diffusion is a stack of easy problems.** Reversing noise in one shot is
  hopeless; reversing one small step is a tractable Gaussian. Chain a few hundred and
  you have a generative model. That decomposition is the entire idea.
- **The closed-form forward jump** (`x_t` from `x_0` in `O(1)`) is what makes
  training practical — you sample a random `t` per example instead of simulating.
- **Predicting `eps` is predicting the score.** `score = -eps / sqrt(1 - abar_t)`,
  so your noise predictor is a density-gradient estimator. Rung 2.2 steers a frozen
  model by *adding another gradient* to it — that only makes sense because of this.
- **The loss plateaus far above zero and that is correct.** At high `t` the noise is
  genuinely unrecoverable. A diffusion loss is not a quality score; judge samples.
- **Diffusion covers modes.** The rare 10% basin survived. This is the practical
  reason diffusion displaced GANs for molecules and structures — coverage matters
  more than sharpness when you are designing candidates.
- **Sampling is slow.** 200 network evaluations for one batch of points. That single
  complaint is the reason the next rung exists.

**Next rep:** `1.4 Flow matching` — the same job in a straighter line. Replace the
noise schedule with a straight path from noise to data, regress a velocity instead
of a noise, and sample with an ODE in a handful of steps rather than 200.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def make_schedule(T=T, b0=1e-4, bT=0.06):
    betas = torch.linspace(b0, bT, T)
    alphas = 1.0 - betas
    return betas, alphas, torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    a = abar[t].unsqueeze(-1)                      # [B, 1] so it broadcasts over [B, 2]
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

def ddpm_loss(model, x0):
    t = torch.randint(0, T, (len(x0),))
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    return F.mse_loss(model(xt, t), noise)

@torch.no_grad()
def p_sample_step(model, xt, t):
    tb = torch.full((len(xt),), t, dtype=torch.long)
    eps = model(xt, tb)
    mean = (xt - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
    if t == 0:
        return mean                                 # no noise on the final step
    return mean + post_var[t].sqrt() * torch.randn_like(xt)

@torch.no_grad()
def sample_loop(model, n, keep=()):
    x = torch.randn(n, 2)
    snaps = {}
    for t in reversed(range(T)):
        x = p_sample_step(model, x, t)
        if t in keep:
            snaps[t] = x.clone()
    return x, snaps

def basin_fractions(pts_deg):
    cen = np.array([b[1] for b in BASINS])
    which = ((pts_deg[:, None, :] - cen[None]) ** 2).sum(-1).argmin(1)
    return np.bincount(which, minlength=len(BASINS)) / len(pts_deg)

print('reference solutions loaded — re-run the checkpoint cells above')